# 02 - AutoGen：微软的多 Agent 协作框架

## 学习目标

- 理解 AutoGen 的核心架构和设计理念
- 掌握 ConversableAgent 和 GroupChat 的用法
- 学习构建多 Agent 协作系统
- 实现代码生成、执行、调试的完整流程

---

## 1. AutoGen 概述

### 1.1 什么是 AutoGen？

**AutoGen** 是微软研究院开发的开源框架，用于构建**多 Agent 对话系统**。它的核心特点是：

- **对话驱动**：Agent 之间通过自然语言对话协作
- **角色定制**：每个 Agent 可以有不同的角色、能力和工具
- **人机协作**：支持人类参与对话，与 Agent 协作
- **代码执行**：内置代码生成和执行能力

### 1.2 AutoGen 的核心概念

| 概念 | 描述 | 类比 |
|------|------|------|
| **Agent** | 可对话的实体，有角色和能力 | 团队成员 |
| **ConversableAgent** | 基础 Agent 类 | 基础员工 |
| **UserProxyAgent** | 代表人类的 Agent | 项目经理 |
| **AssistantAgent** | AI 助手 Agent | 技术专家 |
| **GroupChat** | 多 Agent 群聊 | 团队会议 |
| **Code Execution** | 代码生成和执行 | 开发环境 |

### 1.3 AutoGen 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                     AutoGen 架构                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────┐         ┌─────────────┐                   │
│  │   Human     │◄───────►│ UserProxy   │                   │
│  │  (用户)     │         │   Agent     │                   │
│  └─────────────┘         └──────┬──────┘                   │
│                                 │                           │
│                    ┌────────────┼────────────┐             │
│                    │            │            │             │
│                    ▼            ▼            ▼             │
│              ┌─────────┐ ┌─────────┐ ┌─────────┐         │
│              │Assistant│ │Assistant│ │Assistant│         │
│              │Agent 1  │ │Agent 2  │ │Agent 3  │         │
│              │(Coder)  │ │(Reviewer│ │(Tester) │         │
│              └────┬────┘ └────┬────┘ └────┬────┘         │
│                   │           │           │              │
│                   └───────────┼───────────┘              │
│                               │                          │
│                    ┌──────────┴──────────┐               │
│                    │     GroupChat       │               │
│                    │   (群聊管理器)       │               │
│                    └─────────────────────┘               │
│                                                             │
│  特性：                                                      │
│  • Agent 之间自动协商                                        │
│  • 支持代码生成与执行                                        │
│  • 人类可以随时介入                                          │
│  • 可配置对话终止条件                                        │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. 环境准备

### 2.1 安装 AutoGen



---
## 0. 模型准备：加载 Qwen2.5-7B-Instruct

本 Notebook 使用 **ModelScope** 加载本地 Qwen 模型，替代在线 API 调用。

- **推荐模型**：`Qwen/Qwen2.5-7B-Instruct`（约 15GB 显存）
- **低显存备选**：`Qwen/Qwen2.5-3B-Instruct`（约 6GB 显存）
- 自动检测 GPU / CPU，优先使用 GPU 加速

> 如果没有安装 modelscope 或显存不足，可以使用下方代码中的 **MockLLM 备选方案**。



In [ ]:
# ============================================================
# 安装依赖（首次运行时取消注释）
# ============================================================
# !pip install modelscope torch transformers -q

import torch

# ============================================================
# GPU / CPU 自动检测
# ============================================================
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"检测到 GPU: {GPU_NAME}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("检测到 Apple Silicon GPU (MPS)")
else:
    DEVICE = "cpu"
    print("未检测到 GPU，将使用 CPU（速度较慢）")

# ============================================================
# QwenLLM 封装类
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer

class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测设备
        if device is None:
            device = "cuda" if torch.cuda.is_available() else (
                "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
                else "cpu"
            )
        self.device = device
        print(f"正在加载模型 {model_name}，设备: {device} ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """对话接口"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []

# ============================================================
# 初始化模型
# ============================================================
# 低显存环境可切换为: Qwen/Qwen2.5-3B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

try:
    llm = QwenLLM(model_name=MODEL_NAME)
    USE_REAL_MODEL = True
    print("\n使用真实 Qwen 模型")
except Exception as e:
    print(f"\n模型加载失败: {e}")
    print("将使用 MockLLM 备选方案")
    USE_REAL_MODEL = False

print(f"\n当前设备: {DEVICE}")
print(f"使用真实模型: {USE_REAL_MODEL}")



In [ ]:
# ============================================================
# 无模型时的备选方案：MockLLM
# （仅在上方模型加载失败时使用）
# ============================================================

if not USE_REAL_MODEL:
    class MockLLM:
        """模拟 LLM，用于无模型环境下的教学演示"""

        def __init__(self, model_name="mock-qwen"):
            self.model_name = model_name
            self.messages = []

        def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
            """模拟对话回复"""
            if '你好' in user_message:
                response = "你好！我是 AI 助手，有什么可以帮助你的吗？"
            elif '天气' in user_message:
                response = "我无法获取实时天气信息，建议您查看天气应用。"
            elif 'LangChain' in user_message:
                response = "LangChain 是一个用于开发 LLM 应用的框架，提供了 Chains、Prompts、Memory 等核心组件。"
            elif 'Agent' in user_message:
                response = "Agent 是一种能够自主决策并调用工具的智能体，通常使用 ReAct 模式工作。"
            else:
                response = f"我收到了您的消息：'{user_message[:50]}'。这是一个模拟回复。"

            self.messages.append({"role": "user", "content": user_message})
            self.messages.append({"role": "assistant", "content": response})
            return response

        def reset(self):
            self.messages = []

    llm = MockLLM(model_name="mock-qwen")
    print("已启用 MockLLM 备选方案")

# 测试模型调用
response = llm.chat("你好，请介绍一下自己")
print(f"模型回复: {response}")



In [ ]:
# 安装 AutoGen
# !pip install pyautogen -q

# 验证安装
try:
    import autogen
    print(f"AutoGen 已安装")
except ImportError:
    print("AutoGen 未安装，以下使用模拟实现演示")

print("\n准备完成")



### 2.2 配置 LLM

```python
# AutoGen 配置 LLM
config_list = [
    {
        "model": "gpt-4",
        "api_key": "<your-api-key>",
        "base_url": "https://api.openai.com/v1"
    }
]

# 或者使用国产模型
config_list = [
    {
        "model": "glm-4",
        "api_key": "<your-zhipu-key>",
        "base_url": "https://open.bigmodel.cn/api/paas/v4/"
    }
]
```

---

## 3. 核心组件详解

### 3.1 ConversableAgent

所有 Agent 的基础类，支持发送和接收消息。



In [ ]:
# 模拟 AutoGen Agent 实现（使用 QwenLLM 生成回复）

# --- 无模型时的备选方案：原始 MockConversableAgent（已注释）---
# class MockConversableAgent:
#     def _generate_reply(self, message, sender):
#         if 'Coder' in self.name:
#             return f"[{self.name}] 我来编写代码解决这个问题。..."
#         elif 'Reviewer' in self.name:
#             return f"[{self.name}] 代码审查意见：..."
#         ...

class MockConversableAgent:
    """模拟 ConversableAgent（使用 QwenLLM 生成回复）"""

    def __init__(
        self,
        name,
        system_message="You are a helpful AI assistant.",
        llm_config=None,
        human_input_mode="NEVER",
        max_consecutive_auto_reply=10
    ):
        self.name = name
        self.system_message = system_message
        self.llm_config = llm_config
        self.human_input_mode = human_input_mode
        self.max_consecutive_auto_reply = max_consecutive_auto_reply
        self.chat_history = []
        self.tools = []

    def send(self, message, recipient, request_reply=True):
        """发送消息给另一个 Agent"""
        print(f"\n[{self.name}] -> [{recipient.name}]: {message[:80]}...")

        # 记录消息
        self.chat_history.append({
            'from': self.name,
            'to': recipient.name,
            'message': message
        })

        # 请求回复
        if request_reply:
            recipient.receive(message, self)

    def receive(self, message, sender):
        """接收消息"""
        # 使用 QwenLLM 生成回复
        reply = self._generate_reply(message, sender)

        if reply:
            self.send(reply, sender)

    def _generate_reply(self, message, sender):
        """使用 QwenLLM 生成回复"""
        try:
            # 构建提示词：结合角色信息和消息内容
            prompt = f"你是 {self.name}。{self.system_message}\n\n收到来自 {sender.name if sender else '用户'} 的消息：\n{message}\n\n请以 {self.name} 的身份回复。"
            reply = llm.chat(prompt, system_prompt=self.system_message, max_new_tokens=256)
            return f"[{self.name}] {reply}"
        except Exception as e:
            # 备选：基于角色的简单回复
            if 'Coder' in self.name or '程序员' in self.name:
                return f"[{self.name}] 我来编写代码解决这个问题。\n```python\n# 代码实现\nprint('Hello World')\n```"
            elif 'Reviewer' in self.name or '审查' in self.name or '审查员' in self.name:
                return f"[{self.name}] 代码审查意见：\n1. 建议添加异常处理\n2. 变量命名可以改进"
            elif 'Tester' in self.name or '测试' in self.name or '测试员' in self.name:
                return f"[{self.name}] 测试用例设计：\n- 正常输入测试\n- 边界条件测试\n- 异常输入测试"
            else:
                return f"[{self.name}] 收到，我来处理这个问题。"

    def register_tool(self, tool):
        """注册工具"""
        self.tools.append(tool)

    def __repr__(self):
        return f"MockConversableAgent(name='{self.name}')"

# 创建 Agent
coder = MockConversableAgent(
    name="Coder",
    system_message="你是一个专业的 Python 程序员。"
)

reviewer = MockConversableAgent(
    name="Reviewer",
    system_message="你是一个代码审查专家。"
)

print("Agent 创建完成")
print(f"  - {coder}")
print(f"  - {reviewer}")



### 3.2 两 Agent 对话



In [ ]:
# 模拟两 Agent 对话

print("\n" + "="*60)
print("开始两 Agent 对话")
print("="*60)

# Coder 向 Reviewer 发送消息
coder.send(
    "我编写了一个快速排序算法，请帮我审查。\n```python\ndef quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[len(arr) // 2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)\n```",
    reviewer
)

print("\n" + "="*60)
print("对话统计")
print("="*60)
print(f"总消息数: {len(coder.chat_history)}")
for msg in coder.chat_history:
    print(f"  [{msg['from']} → {msg['to']}] {msg['message'][:50]}...")



### 3.3 GroupChat（群聊）

GroupChat 允许多个 Agent 在一个群聊中协作。



In [ ]:
# 模拟 GroupChat

class MockGroupChat:
    """模拟 GroupChat"""

    def __init__(
        self,
        agents,
        messages=[],
        max_round=10
    ):
        self.agents = agents
        self.messages = messages
        self.max_round = max_round

    def select_speaker(self, last_speaker):
        """选择下一个发言者"""
        # 轮询选择
        if last_speaker is None:
            return self.agents[0]

        idx = self.agents.index(last_speaker)
        next_idx = (idx + 1) % len(self.agents)
        return self.agents[next_idx]

    def run(self, initial_message):
        """运行群聊"""
        print(f"\n{'='*60}")
        print("开始 GroupChat")
        print(f"{'='*60}")
        print(f"参与者: {[a.name for a in self.agents]}")
        print(f"最大轮数: {self.max_round}\n")

        # 添加初始消息
        self.messages.append({
            'speaker': 'User',
            'message': initial_message
        })

        print(f"[User] {initial_message}\n")

        current_speaker = None

        for round_num in range(self.max_round):
            # 选择发言者
            speaker = self.select_speaker(current_speaker)
            current_speaker = speaker

            # 生成回复（使用 QwenLLM）
            reply = speaker._generate_reply(
                self.messages[-1]['message'],
                None
            )

            self.messages.append({
                'speaker': speaker.name,
                'message': reply
            })

            print(f"[{speaker.name}] {reply}\n")

            # 检查是否终止
            if 'TERMINATE' in reply or '完成' in reply:
                print("对话终止")
                break

        return self.messages

class MockGroupChatManager:
    """模拟 GroupChatManager"""

    def __init__(self, groupchat, llm_config=None):
        self.groupchat = groupchat
        self.llm_config = llm_config

    def initiate_chat(self, agent, message):
        """发起群聊"""
        return self.groupchat.run(message)

# 创建多个 Agent
coder = MockConversableAgent(
    name="程序员",
    system_message="你是专业程序员，负责编写代码。"
)

reviewer = MockConversableAgent(
    name="审查员",
    system_message="你是代码审查专家，负责检查代码质量。"
)

tester = MockConversableAgent(
    name="测试员",
    system_message="你是测试专家，负责设计测试用例。"
)

# 创建 GroupChat
groupchat = MockGroupChat(
    agents=[coder, reviewer, tester],
    max_round=6
)

manager = MockGroupChatManager(groupchat)

print("GroupChat 初始化完成")



In [ ]:
# 运行群聊
messages = manager.initiate_chat(
    agent=coder,
    message="请团队协作完成一个用户登录功能的开发"
)

print(f"\n群聊统计")
print(f"总消息数: {len(messages)}")



---

## 4. 实际使用 AutoGen

### 4.1 基础用法



In [ ]:
autogen_example = '''
# 实际 AutoGen 代码示例

import autogen

# 配置 LLM
config_list = [
    {
        "model": "gpt-4",
        "api_key": "<your-api-key>"
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0.7
}

# 创建 Agent
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message="你是一个 helpful 的 AI 助手。"
)

user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False
    }
)

# 开始对话
user_proxy.initiate_chat(
    assistant,
    message="写一个 Python 函数计算斐波那契数列"
)
'''

print("AutoGen 基础用法：")
print("=" * 50)
print(autogen_example)



### 4.2 GroupChat 用法



In [ ]:
groupchat_example = '''
# GroupChat 实际用法

import autogen

# 创建多个 Agent
coder = autogen.AssistantAgent(
    name="coder",
    system_message="你是专业程序员。编写代码后回复 TERMINATE。"
)

reviewer = autogen.AssistantAgent(
    name="reviewer",
    system_message="你是代码审查专家。审查后回复 TERMINATE。"
)

tester = autogen.AssistantAgent(
    name="tester",
    system_message="你是测试专家。设计测试后回复 TERMINATE。"
)

# 创建 GroupChat
groupchat = autogen.GroupChat(
    agents=[coder, reviewer, tester],
    messages=[],
    max_round=12,
    speaker_selection_method="round_robin"
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# 开始群聊
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER"
)

user_proxy.initiate_chat(
    manager,
    message="请团队协作完成一个用户登录功能"
)
'''

print("GroupChat 实际用法：")
print("=" * 50)
print(groupchat_example)



### 4.3 代码执行配置



In [ ]:
code_execution_example = '''
# 代码执行配置

# 方式 1：本地执行（不安全，仅开发使用）
code_execution_config = {
    "work_dir": "coding",  # 工作目录
    "use_docker": False,   # 不使用 Docker
    "last_n_messages": 3   # 查看最近 3 条消息中的代码
}

# 方式 2：Docker 执行（推荐）
code_execution_config = {
    "work_dir": "coding",
    "use_docker": True,    # 使用 Docker 隔离
    "docker_image": "python:3.11"
}

# 方式 3：禁用代码执行
code_execution_config = False

user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    code_execution_config=code_execution_config
)
'''

print("代码执行配置：")
print("=" * 50)
print(code_execution_example)



---

## 5. AutoGen 最佳实践

### 5.1 Agent 设计原则

| 原则 | 说明 | 示例 |
|------|------|------|
| **角色明确** | 每个 Agent 有清晰的职责 | Coder 只写代码，Reviewer 只审查 |
| **终止条件** | 明确何时结束对话 | 回复 TERMINATE |
| **上下文管理** | 控制历史消息长度 | last_n_messages |
| **错误处理** | 处理工具执行失败 | try-except |

### 5.2 常见模式

```
模式 1：两 Agent 协作
    UserProxy ↔ Assistant
    （最简单，适合大多数场景）

模式 2：三 Agent 协作
    UserProxy ↔ Coder ↔ Reviewer
    （代码生成 + 审查）

模式 3：群聊协作
    UserProxy + GroupChat(Coder, Reviewer, Tester)
    （复杂项目，多角色协作）

模式 4：分层协作
    Manager Agent
        ├── Coder Agent
        ├── Reviewer Agent
        └── Tester Agent
    （层级管理，适合大型项目）
```

---

## 6. 小结

### 核心要点

1. **AutoGen** 是微软的多 Agent 对话框架
2. **核心概念**：ConversableAgent、GroupChat、Code Execution
3. **对话驱动**：Agent 通过自然语言协作完成任务
4. **角色定制**：每个 Agent 可以有不同的系统提示和能力
5. **代码执行**：内置代码生成和执行能力

### 下一步

- [03_crewai_collaboration.ipynb](03_crewai_collaboration.ipynb) - 学习 CrewAI 协作框架
- [04_llamindex_rag_agent.ipynb](04_llamindex_rag_agent.ipynb) - 探索 LlamaIndex RAG Agent

---

## 参考资源

- [AutoGen 官方文档](https://microsoft.github.io/autogen/)
- [AutoGen GitHub](https://github.com/microsoft/autogen)
- [AutoGen 论文](https://arxiv.org/abs/2308.08155)

